# 用 Prophet 做 VIC 零售需求预测：从业务问题到预测系统

本 Notebook 按博客九章逻辑组织：引言 → 问题与案例 → 建模理论 → 输入与配置 → 求解与解释 → 评估诊断 → MLOps → 模型拓展比较 → 实施建议。具体目录可通过 Jupyter 的 Table of Contents 导航。

先运行 `prophet-retail-data-generator.ipynb`，或使用下载包中的 `data/teaching/`，再执行本 Notebook 的 Restart Kernel and Run All。三套主案例数据与 VIC 假日表通过 CSV 加载，原始文件保持原销售标签。

主案例为作者设定的墨尔本高频食品：起始需求 85 件、周末增量 16 件、年振幅 14 件、营业假日增量 28 件、两周一次八折促销。2023—2025 年共 1096 天，最后 28 天为测试期；末段指标检查圣诞节和节礼日，秋冬表现需要另外查看对应回测窗口。

本 Notebook 保留 Prophet、LightGBM、基线、参数对照、区间、诊断和附录拟合接口。第六、八章的独立机制实验使用各自种子另造小数据，用于隔离参数作用；三套零售 CSV 统一由数据生成 Notebook 管理。教学结果不代表品牌实测规律，ARIMA、XGBoost 与神经网络仅作机制比较，未新增性能排名。

## 一、引言与环境准备

预测每天的需求之前，先明确补货决策需要的粒度、提前期与可用输入。这里用一条序列跑通流程，后面讨论批量预测与模型扩展。

### 1.1 环境与运行方式

建议使用独立 Python 环境，安装配套 `pyproject.toml`，然后用 JupyterLab 或 VS Code 打开本文件。若已经进入 Notebook，也可取消下一单元第一行注释进行安装；安装后必要时重启 kernel。

第一遍先按顺序执行全部单元。图像直接嵌入输出，模型和结果文件写入当前目录的 `runs/<run-id>/`。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
import sys
import os
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from io import BytesIO
import matplotlib.pyplot as plt
from IPython.display import display


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + "-prophet"
OUTPUT_DIR = REPO_ROOT / "runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)


def save_plot(figure, path):
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=140, bbox_inches="tight")
    Path(path).write_bytes(buffer.getvalue())


print("Python:", sys.version.split()[0])
display(
    {
        pkg: version(pkg)
        for pkg in ["prophet", "pandas", "numpy", "holidays", "cmdstanpy"]
    }
)
import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.make_holidays import make_holidays_df
import logging

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
np.random.seed(42)  # 固定默认预测区间的模拟随机性。

## 二、问题评估、数据与案例

业务问题评估（Problem Assessment）确定需求或销量口径、门店商品粒度、28 天窗口与验收方式；数据与业务知识获取（Data and Knowledge Acquisition）确定销售、库存、营业日历和当时可见的促销计划。开发与原型（Development / Prototype）完成一个可复现的模型；完整系统（Complete System）接入编排、验证、回退与版本保存。评估与验证（Evaluation and Validation）贯穿开发，部署（Deployment）与集成（Integration）连接预测到补货，维护运行或交接（Maintenance and Operation / Handover）落实持续评估与责任。

本 Notebook 验证教学数据与模型实验，第七章演示保存和交付接口。生产容量、真实系统集成和实际业务收益需要在对应环境进一步验证。

### 2.1 从 CSV 读取主案例与 VIC 假日表

`DATA_DIR` 是相对于内核工作目录的输入目录。缺少文件时会显示明确路径，请先运行生成器或解压完整包。`conditional_mean` 是无噪声均值，`latent_demand` 是含噪声潜在需求，两列仅用于诊断；模型输入显式选择 `ds`、`y`、`discount`。

缺货日销售受限，闭店为零。训练副本屏蔽这些标签，原 CSV 与 `data` 保留原值。预测起点为 2025-12-03，最后 28 天仅用于最终评价。

In [ ]:
def read_retail_csv(path, feature_cols=("discount",), require_y=True):
    frame = pd.read_csv(path)
    required = ["ds", *feature_cols] + (["y"] if require_y else [])
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    frame["ds"] = pd.to_datetime(frame["ds"], errors="raise")
    if frame["ds"].isna().any() or frame["ds"].dt.tz is not None:
        raise ValueError("ds must contain valid, timezone-free business dates")
    if frame["ds"].duplicated().any():
        raise ValueError("Aggregate or resolve duplicate dates before fitting")
    for col in feature_cols:
        frame[col] = pd.to_numeric(frame[col], errors="raise")
        if not np.isfinite(frame[col].to_numpy(dtype=float)).all():
            raise ValueError(f"Invalid feature values in {col}")
    if require_y:
        frame["y"] = pd.to_numeric(frame["y"], errors="raise")
        if np.isinf(frame["y"].to_numpy(dtype=float)).any():
            raise ValueError("Infinite y is not allowed")
        valid = frame.dropna(subset=["y"])
        if valid["ds"].nunique() < 2:
            raise ValueError("At least two distinct valid training dates are required")
    return frame.sort_values("ds").reset_index(drop=True)


# 真实数据使用示例：取消注释前先准备文件和合适的日历。
# own_history = read_retail_csv('history.csv')
# own_plan = read_retail_csv('future_plan.csv', require_y=False)

In [ ]:
DATA_DIR = REPO_ROOT / "data" / "teaching"
HORIZON_DAYS = 28
TIME_ORIGIN = pd.Timestamp("2023-01-01")
required_paths = [
    DATA_DIR / name
    for name in [
        "grocery_scenario_data.csv",
        "wine_scenario_data.csv",
        "garden_scenario_data.csv",
        "vic_holidays.csv",
    ]
]
missing_paths = [str(p.resolve()) for p in required_paths if not p.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "请先运行 prophet-retail-data-generator.ipynb 或解压教学数据：\n"
        + "\n".join(missing_paths)
    )


def read_scenario_csv(path, feature_cols):
    frame = read_retail_csv(path, feature_cols=feature_cols)
    required = {"is_open", "stockout", "conditional_mean", "latent_demand"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{path} 缺少字段：{sorted(missing)}")
    for col in ["is_open", "stockout"]:
        if not pd.api.types.is_bool_dtype(frame[col]):
            raise ValueError(f"{path} 的 {col} 需要 True/False 布尔值")
    if frame.y.isna().any() or (frame.y < 0).any():
        raise ValueError("生成器 CSV 需要完整、非负的原始销售标签")
    if not np.equal(frame.y, np.rint(frame.y)).all():
        raise ValueError("本教学数据的销售单位为整数件")
    if not (frame.loc[~frame.is_open, "y"] == 0).all():
        raise ValueError("闭店销售应为零")
    if (frame.stockout & ~frame.is_open).any():
        raise ValueError("本教学规则的缺货标记仅用于营业日")
    if not frame.ds.diff().dropna().eq(pd.Timedelta(days=1)).all():
        raise ValueError("教学 CSV 应保留连续的日历日期")
    if not np.isfinite(frame[["conditional_mean", "latent_demand"]].to_numpy()).all():
        raise ValueError("诊断真实值需要有限数值")
    return frame


data = read_scenario_csv(DATA_DIR / "grocery_scenario_data.csv", ("discount",))
vic_holidays = pd.read_csv(DATA_DIR / "vic_holidays.csv", parse_dates=["ds"])
assert {"ds", "holiday", "lower_window", "upper_window"} <= set(vic_holidays)
assert vic_holidays.ds.notna().all() and vic_holidays.holiday.notna().all()
vic_holidays = vic_holidays.sort_values(["ds", "holiday"]).reset_index(drop=True)
ds = pd.DatetimeIndex(data.ds)
t = ((ds - TIME_ORIGIN) / pd.Timedelta(days=1)).to_numpy()
cutoff = data.ds.iloc[-HORIZON_DAYS - 1]
train = data.loc[data.ds <= cutoff].copy()
test = data.loc[data.ds > cutoff].copy()
train.loc[train.stockout | ~train.is_open, "y"] = np.nan
assert len(test) == HORIZON_DAYS and train.ds.max() < test.ds.min()
assert train.loc[~train.is_open | train.stockout, "y"].isna().all()
assert data.y.notna().all()

# 仅为第八章的多业务特征独立实验保留已知生成设定；主案例标签始终来自 CSV。
discount = data.discount.to_numpy()
weekend = (ds.dayofweek >= 5).astype(float)
annual = 14 * np.sin(2 * np.pi * (t - 15) / 365.25)
holiday_uplift = 28 * ds.isin(vic_holidays.ds.unique()).astype(float)


def sales_from_open_forecast(frame, raw_prediction):
    open_mask = (
        frame["is_open"].to_numpy(bool)
        if "is_open" in frame
        else np.ones(len(frame), bool)
    )
    return np.where(open_mask, np.maximum(np.asarray(raw_prediction, float), 0), 0)


display(data.head())
display(vic_holidays.loc[vic_holidays.ds.dt.year == 2025])
display(
    {
        "csv_path": str((DATA_DIR / "grocery_scenario_data.csv").resolve()),
        "training_rows": len(train),
        "valid_labels": int(train.y.notna().sum()),
        "cutoff": str(cutoff.date()),
        "test_rows": len(test),
    }
)

### 2.2 检查销售、潜在需求与时间划分

无噪声均值用于核对生成机制，销售还受到噪声、库存和营业状态影响。阴影右侧为最终测试期。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), constrained_layout=True)
for ax, frame in zip(axes, [data, data.loc[data.ds >= "2025-11-01"]]):
    ax.plot(
        frame.ds,
        frame.conditional_mean,
        label="Noise-free open-day demand",
        color="steelblue",
    )
    ax.plot(frame.ds, frame.y, label="Observed sales", color="grey", alpha=0.65)
    limited = frame.loc[frame.stockout & frame.is_open]
    closed = frame.loc[~frame.is_open]
    ax.scatter(limited.ds, limited.y, marker="x", color="darkorange", label="Stockout")
    ax.scatter(closed.ds, closed.y, marker="s", color="purple", label="Closed")
    ax.axvline(cutoff, linestyle="--", color="red", label="Training cutoff")
    ax.axvspan(
        test.ds.min(), test.ds.max(), alpha=0.12, color="orange", label="28-day holdout"
    )
    ax.set_ylabel("Units")
    ax.legend(fontsize=8)
save_plot(fig, OUTPUT_DIR / "teaching_data_overview.png")
plt.show()

## 三、建模理论：趋势、周期与事件

Prophet 将需求写为趋势、季节性、假日与业务回归贡献的合计。加法主案例为：

$$y(t)=g(t)+s(t)+h(t)+\mathbf{x}_t^\top\boldsymbol\beta+\varepsilon_t.$$

$t$ 是日期对应的时间坐标，$y(t)$ 是标签，$g(t)$ 是趋势，$s(t)$ 是周期，$h(t)$ 是事件贡献，$\mathbf x_t$ 是业务特征向量，$\boldsymbol\beta$ 是业务系数，$\varepsilon_t$ 是误差；$\top$ 表示转置，内积表示逐项乘积之和。

### 3.1 从周周期的三阶配置观察六列傅里叶特征

`period=7` 定义每 7 天重复，`fourier_order=3` 生成 3 对正弦/余弦、共 6 列。下方只构造确定性特征；系数在第五章联合拟合后检查。年周期的 5 阶对应 10 列，是需要验证的表达复杂度。

In [ ]:
fourier_dates = pd.Series(pd.date_range("2025-12-01", periods=7), name="ds")
days_since_epoch = (fourier_dates - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
fourier_demo = pd.DataFrame({"ds": fourier_dates})
for order in range(1, 4):
    angle = 2 * np.pi * order * days_since_epoch / 7
    fourier_demo[f"sin_{order}"] = np.sin(angle)
    fourier_demo[f"cos_{order}"] = np.cos(angle)
basis = fourier_demo.drop(columns="ds").to_numpy()
assert basis.shape == (7, 6) and np.linalg.matrix_rank(basis) == 6
assert np.linalg.matrix_rank(np.column_stack([np.ones(7), basis])) == 7
fourier_demo["ideal_weekly_units"] = 16 * (
    (fourier_dates.dt.dayofweek >= 5).astype(float) - 2 / 7
)
display(fourier_demo)
print("6 个独立周期自由度，加上整体水平，可表达 7 个固定星期位置。")

## 四、数据处理与参数配置

趋势使用训练起点与跨度，周期使用固定日历相位。先检查输入，再指定模型可以学习的结构；内部拟合系数在下一章检查。

### 4.1 日期字符串怎样变成数值时间

`ds` 会被解析成时间戳。趋势使用从训练起点开始的归一化时间，季节性使用固定日历时间生成周期函数，两者不能混用。

下面只用三天说明趋势缩放：训练跨度为两天，因此第一、二、三天分别为 0、0.5、1；第四天是 1.5。它不是重新在未来数据上缩放，也不是简单使用行号。

In [ ]:
import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.make_holidays import make_holidays_df

dates = pd.to_datetime(["2023-01-01", "2023-01-02", "2023-01-03", "2023-01-04"])
span = dates[2] - dates[0]
date_demo = pd.DataFrame({"ds": dates})
date_demo["trend_time_demo"] = (dates - dates[0]) / span
date_demo["days_since_1970"] = (dates - pd.Timestamp("1970-01-01")) / pd.Timedelta(
    days=1
)
angle = 2 * np.pi * date_demo["days_since_1970"] / 7
date_demo["weekly_sin_1"] = np.sin(angle)
date_demo["weekly_cos_1"] = np.cos(angle)
display(date_demo)
np.testing.assert_allclose(date_demo["trend_time_demo"], [0, 0.5, 1, 1.5])

### 4.2 假日窗口怎样编码

教学检查使用 Prophet 1.4.0 的内部 `make_holiday_features()` 方法，观察圣诞节前一天和当天分别如何编码。主模型仍使用原先的当天窗口，不会被这个检查改变。

同一名称的不同年份共享相同特征；不同相对日期使用不同列。`_delim_` 等名称是实现细节，生产流程应依赖公共 API。

In [ ]:
holiday_window_demo = vic_holidays.loc[
    vic_holidays["holiday"].eq("Christmas Day")
].copy()
holiday_window_demo["lower_window"] = -1
holiday_window_demo["upper_window"] = 0
probe = pd.Series(pd.date_range("2025-12-23", "2025-12-27", freq="D"))
encoder = Prophet(holidays=holiday_window_demo)
encoded, _, _ = encoder.make_holiday_features(probe, holiday_window_demo)
display(pd.concat([probe.rename("ds"), encoded], axis=1))
assert encoded.sum().sum() == 2

### 4.3 模型配置与拟合

`build_model()` 显式添加周、年周期、VIC 假日和折扣。

- `growth='linear'`：允许基础需求的增长速率改变。
- `n_changepoints` 与 `changepoint_range`：安排候选变点；不是已经确认的业务变化。
- `changepoint_prior_scale`：控制趋势的灵活程度。
- 关闭自动周、年周期后手动添加，便于理解阶数。
- `seasonality_mode` 和 `holidays_mode`：本例按额外件数表达贡献。
- 两类 `prior_scale`：约束系数，不能按“一件销量”解读。
- `mcmc_samples=0`：MAP 路径；`uncertainty_samples` 用于预测区间模拟。

供给受限日的训练标签设为缺失，不表示已经恢复潜在需求。假日与折扣系数的统计关联也不能直接解释为因果效应。

In [ ]:
def build_model():
    model = Prophet(
        growth="linear",  # 基础需求允许缓慢增长或下降。
        n_changepoints=25,  # 候选斜率变化位置，并非确定的业务变更次数。
        changepoint_range=0.80,  # 在历史前 80% 范围内安排候选变点。
        changepoint_prior_scale=0.05,  # 控制趋势灵活度，减少把短期促销当成长期增长。
        daily_seasonality=False,  # 日级销量不能识别一天内的小时规律。
        weekly_seasonality=False,  # 下方手动添加周周期，避免重复配置。
        yearly_seasonality=False,  # 下方手动添加年周期，明确模型复杂度。
        seasonality_mode="additive",  # 周期波动按额外件数表达。
        seasonality_prior_scale=1.0,  # 限制周期曲线过度放大，具体值需要回测。
        holidays=vic_holidays,  # 传入 AU/VIC 的历史与未来假日日期。
        holidays_mode="additive",  # 假日影响按额外件数表达，系数可正可负。
        holidays_prior_scale=1.0,  # 收缩假日系数，避免少数年份支撑过强结论。
        mcmc_samples=0,  # 使用 MAP 优化，作为批量训练的起点。
        interval_width=0.80,  # 请求模型假设下的 80% 预测区间。
        uncertainty_samples=1000,  # 用于区间计算的模拟样本数，不是训练样本数。
    )
    model.add_seasonality(name="weekly", period=7, fourier_order=3)
    model.add_seasonality(name="yearly", period=365.25, fourier_order=5)
    model.add_regressor(
        "discount",
        prior_scale=1.0,
        standardize=False,
        mode="additive",
    )
    return model


model = build_model()
model.fit(train[["ds", "y", "discount"]], seed=42)

# 真实系统从截至 cutoff 可见的计划快照读取 discount。
future = test[["ds", "discount"]].copy()
forecast = model.predict(future)

result = test.merge(
    forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
    on="ds",
    how="left",
    validate="one_to_one",
)

# 保留原始模型输出；业务非负规则使用独立字段。
result["forecast_units"] = sales_from_open_forecast(result, result["yhat"])
print(result[["ds", "y", "forecast_units"]].head())


display(result[["ds", "y", "stockout", "discount", "yhat", "forecast_units"]].head())
assert np.isfinite(forecast["yhat"]).all()
assert (result.loc[~result["is_open"], "forecast_units"] == 0).all()

### 4.4 检查训练后保存的输入变换

`model.history` 保存有效训练标签的内部表示。`t` 是模型使用的归一化趋势时间，`y_scaled` 是目标缩放结果。本例线性、absmax 配置不使用下限偏移，所以 `y_scaled = y / y_scale`。

检查内部属性是教学用途；生产代码可以直接使用 `fit()` 和 `predict()`。缺失标签不被自动补零，缺货需求也不被自动修复。

In [ ]:
display(model.history[["ds", "t", "y", "y_scaled", "discount"]].head())
display(
    {
        "training_start": str(model.start),
        "training_span": str(model.t_scale),
        "y_scale": model.y_scale,
        "future_first_normalized_t": float(
            (future["ds"].iloc[0] - model.start) / model.t_scale
        ),
    }
)
np.testing.assert_allclose(
    model.history["y_scaled"], model.history["y"] / model.y_scale
)

probe_features = model.make_seasonality_features(
    model.history["ds"].head(), period=7, series_order=3, prefix="weekly"
)
display(probe_features)
assert probe_features.shape[1] == 6

## 五、优化求解与模型可解释性

MAP（Maximum a Posteriori Estimation，最大后验估计）联合估计趋势、变点变化、完整特征系数与噪声尺度：

$$\hat\theta_{\mathrm{MAP}}=\operatorname*{arg\,max}_{\theta}[\log p(\mathcal D\mid\theta)+\log p(\theta)].$$

$\theta$ 是未知参数集合，$\mathcal D$ 是有效历史标签及给定输入，$p(\mathcal D\mid\theta)$ 是评价参数如何解释观测的似然，$p(\theta)$ 是参数先验密度，$\log$ 是自然对数，$\operatorname*{arg\,max}$ 返回使目标最大的参数。`mcmc_samples=0` 使用 MAP；本文固定版本默认通常采用 L-BFGS。详细先验与负对数目标推导见博客第五章。

下面从组件曲线回到拟合参数，再重建目标日预测。组件提供统计分解，因果判断还需业务证据。

### 5.1 总预测与组件图

为了同时展示历史拟合和未来预测，下面对全部教学日期计算组件。截止线右侧才是测试期预测；左侧的拟合值不能作为泛化评估结果。

`trend` 是长期趋势；`weekly` 与 `yearly` 是重复周期的贡献，不是总销量，也不是观测的简单分组均值。`holidays` 汇总事件效应；`extra_regressors_additive` 在主模型中表示折扣贡献。

加法模式下，组件值按日级件数解释；负值表示减少预测。如果周六组件为 +18，不代表周六总销量为 18。实际本例理想周末约 +11.43、平日约 -4.57，差值为生成数据时设置的 16 件；实际值由联合拟合决定；因为周组件表达的是相对周期基准的偏移。年曲线不是同比增速。乘法模式下，组件按趋势的比例解释。

组件对应假设正常营业时的需求曲线；闭店不属于可学习的假日增量。圣诞节的原始 yhat 可以非零，已知闭店日销售预测才强制为零。图同时保存这两类输出，勿把组件图理解成闭店日实际销量。

In [ ]:
all_inputs = data[["ds", "discount"]].copy()
all_forecast = model.predict(all_inputs)
fig = model.plot(all_forecast)
fig.axes[0].axvline(cutoff, color="red", linestyle="--", label="Forecast origin")
fig.axes[0].set_title("Daily retail forecast: training fit and 28-day holdout")
fig.axes[0].set_ylabel("Units per day")
fig.axes[0].legend()
save_plot(fig, OUTPUT_DIR / "forecast.png")
plt.show()

component_fig = model.plot_components(all_forecast)
for ax in component_fig.axes:
    if ax.get_xlabel() == "ds":
        ax.axvline(cutoff, color="red", linestyle="--")
save_plot(component_fig, OUTPUT_DIR / "components.png")
plt.show()
local_forecast = model.predict(
    data.loc[data["ds"] >= cutoff - pd.Timedelta(days=14), ["ds", "discount"]]
)
local_fig = model.plot_components(local_forecast)
for ax in local_fig.axes:
    if ax.get_xlabel() == "ds":
        ax.axvline(cutoff, color="red", linestyle="--")
save_plot(local_fig, OUTPUT_DIR / "components_local.png")
plt.show()

# 区分原始营业需求与营业日历约束后的销售；闭店日有确定的零销售。
fig, ax = plt.subplots(figsize=(12, 4))
local = data.loc[data["ds"] >= cutoff - pd.Timedelta(days=14)].copy()
local_prediction = model.predict(local[["ds", "discount"]])
ax.plot(local["ds"], local["y"], "o-", label="Observed sales", color="black")
ax.plot(
    local["ds"],
    local_prediction["yhat"],
    "--",
    label="Raw open-day demand",
    color="steelblue",
)
ax.plot(
    local["ds"],
    sales_from_open_forecast(local, local_prediction["yhat"]),
    label="Sales after opening calendar",
    color="darkorange",
)
ax.axvline(cutoff, color="red", linestyle="--", label="Forecast origin")
for d in local.loc[~local["is_open"], "ds"]:
    ax.axvline(d, color="grey", alpha=0.5)
ax.set(
    title="Demand components and structural zero sales are different", ylabel="Units"
)
ax.legend()
fig.autofmt_xdate()
save_plot(fig, OUTPUT_DIR / "opening_calendar_sales.png")
plt.show()

### 5.2 用数值核对组件

`additive_terms` 已经包含周、年、假日和加法回归贡献，不能再重复加一次。本例全部采用加法模式，因此 `yhat = trend + additive_terms`。

周规律在一周网格上检查；假日数值可以在圣诞节与节礼日附近检查。图形中的周期不是因果证据，模型可能把相关因素的影响分配给不同模块。

In [ ]:
component_cols = [
    "ds",
    "trend",
    "weekly",
    "yearly",
    "holidays",
    "extra_regressors_additive",
    "additive_terms",
    "yhat",
]
display(forecast[component_cols].head())
np.testing.assert_allclose(
    forecast["yhat"], forecast["trend"] + forecast["additive_terms"]
)
np.testing.assert_allclose(
    forecast["additive_terms"],
    forecast["weekly"]
    + forecast["yearly"]
    + forecast["holidays"]
    + forecast["extra_regressors_additive"],
)

week_probe = pd.DataFrame(
    {"ds": pd.date_range("2025-12-01", periods=7), "discount": 0.0}
)
week_components = model.predict(week_probe)
week_components["weekday"] = week_components["ds"].dt.day_name()
display(week_components[["ds", "weekday", "weekly"]])
display(
    all_forecast.loc[
        all_forecast["ds"].between("2025-12-23", "2025-12-27"), component_cols
    ]
)

### 5.3 内部参数与原单位系数

Stan 保存 `k`、`m`、`delta`、`beta` 和 `sigma_obs`。源码的 `beta` 同时包含季节性、事件和业务系数，不是只包含折扣。

原始参数处在内部缩放尺度；不要直接把内部 `k` 解读为每天增加的件数。`regressor_coefficients()` 用于查看业务回归系数的原目标尺度；MAP 下的系数上下限不提供完整后验不确定性。

In [ ]:
from prophet.utilities import regressor_coefficients

display(
    pd.DataFrame(
        [
            {"parameter": key, "array_shape": str(value.shape)}
            for key, value in model.params.items()
        ]
    )
)
display(regressor_coefficients(model))
# 配置与拟合参数分开查看；以下内部接口针对 Prophet 1.4.0。
display(pd.DataFrame(model.seasonalities).T)
trend_parameters = pd.DataFrame(
    {
        "changepoint": model.changepoints.to_numpy(),
        "delta_internal": model.params["delta"].mean(axis=0),
    }
)
display(trend_parameters)
training_features, _, training_components, _ = model.make_all_seasonality_features(
    model.history
)
beta_mean = model.params["beta"].mean(axis=0)
coefficient_table = pd.DataFrame(
    {
        "feature": training_features.columns,
        "beta_internal": beta_mean,
        "coefficient_units": beta_mean * model.y_scale,  # 本例全加法。
    }
)
display(coefficient_table)
print("y_scale:", model.y_scale)
coefficient_table.to_csv(OUTPUT_DIR / "prophet_coefficients.csv", index=False)

### 5.4 从内部参数重建周六与节礼日预测

本例为 MAP、全加法、线性、absmax、无下限偏移。内部特征顺序必须与 `beta` 一致；输入也必须使用拟合时保存的变换。第 8 节的系数表与这里的特征乘积共同解释组件曲线。此处不把公式推广到乘法、逻辑斯蒂或 MCMC 均值组合。


In [ ]:
# 用当前模型的内部预处理，沿用历史保存的时间和业务变换。
probe_dates = data.loc[
    data["ds"].isin(pd.to_datetime(["2025-12-06", "2025-12-26"])), ["ds", "discount"]
].reset_index(drop=True)
prepared = model.setup_dataframe(probe_dates.copy(), initialize_scales=False)
X, _, component_mask, _ = model.make_all_seasonality_features(prepared)
assert list(X.columns) == list(training_features.columns)
k = float(model.params["k"].mean())
m = float(model.params["m"].mean())
delta = model.params["delta"].mean(axis=0)
time = prepared["t"].to_numpy()
hinges = np.maximum(time[:, None] - model.changepoints_t[None, :], 0)
trend_internal = k * time + m + hinges @ delta
feature_contributions = X.to_numpy() * beta_mean[None, :] * model.y_scale
rebuilt = model.y_scale * trend_internal + feature_contributions.sum(axis=1)
probe_prediction = model.predict(probe_dates)
np.testing.assert_allclose(rebuilt, probe_prediction["yhat"], rtol=1e-10, atol=1e-8)
reconstruction = pd.DataFrame(
    {"ds": prepared["ds"], "trend": model.y_scale * trend_internal}
)
for name in ["weekly", "yearly", "holidays", "discount"]:
    indices = component_mask[name].to_numpy().astype(bool)
    reconstruction[name] = feature_contributions[:, indices].sum(axis=1)
reconstruction["rebuilt_yhat"] = rebuilt
reconstruction["prophet_yhat"] = probe_prediction["yhat"].to_numpy()
display(reconstruction)
weekly_indices = component_mask["weekly"].to_numpy().astype(bool)
weekly_terms = pd.DataFrame(
    {
        "feature": X.columns[weekly_indices],
        "feature_value": X.iloc[0].to_numpy()[weekly_indices],
        "beta_internal": beta_mean[weekly_indices],
        "contribution_units": feature_contributions[0, weekly_indices],
    }
)
display(weekly_terms)
np.testing.assert_allclose(
    weekly_terms["contribution_units"].sum(), probe_prediction["weekly"].iloc[0]
)
reconstruction.to_csv(OUTPUT_DIR / "prediction_reconstruction.csv", index=False)
weekly_terms.to_csv(OUTPUT_DIR / "weekly_term_contributions.csv", index=False)

# 标准化价格回归：原单位系数仍需乘以偏离训练均值的数量。
# extra_model 在第 8.1 节才创建，相关核对放在该节。

## 六、模型评估、诊断与参数选择

只用训练历史选择配置，最终测试期单独评价。主模型与选参模型使用不同变量，区间和残差仍回到原主模型。

### 6.1 指标与季节朴素基线

WAPE 是绝对误差总和除以实际销量总和；bias 用带符号误差，正值表示高估。分母为零或没有有效样本时不定义。

我们只在未缺货的测试日上比较观测销量。这不等于验证了缺货日潜在需求。基线在预测发起时固定，用每个星期位置最后一次有效历史观测，后续不读取测试实际值。

主要需求指标只评价营业且未缺货日期；闭店销售为已知零，另报告包含闭店日的销售指标，避免以容易预测的零值降低平均误差。

In [ ]:
def demand_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if len(actual) == 0 or actual.sum() <= 0:
        return {"wape": np.nan, "bias": np.nan}
    return {
        "wape": np.abs(actual - predicted).sum() / actual.sum(),
        "bias": (predicted - actual).sum() / actual.sum(),
    }


# 每个星期位置使用训练集中最近一次有效观测。
valid_train = train.dropna(subset=["y"]).copy()
valid_train["weekday"] = valid_train["ds"].dt.dayofweek
last_weekday = valid_train.sort_values("ds").groupby("weekday")["y"].last()
result["seasonal_naive"] = sales_from_open_forecast(
    result, result["ds"].dt.dayofweek.map(last_weekday)
)

# 真实业务中需求未知：这里只在未缺货日比较销售观测。
observed = result.loc[~result["stockout"] & result["is_open"]]
print("Prophet:", demand_metrics(observed["y"], observed["forecast_units"]))
print("Seasonal naive:", demand_metrics(observed["y"], observed["seasonal_naive"]))

metric_table = pd.DataFrame(
    {
        "Prophet": demand_metrics(observed["y"], observed["forecast_units"]),
        "Seasonal naive": demand_metrics(observed["y"], observed["seasonal_naive"]),
    }
).T
display(metric_table)

In [ ]:
def comparison_metrics(frame, prediction_column):
    # 同一评估日期；所有指标均用非负业务预测。
    scored = frame.loc[~frame["stockout"] & frame["is_open"]].copy()
    actual = scored["y"].to_numpy(dtype=float)
    predicted = scored[prediction_column].to_numpy(dtype=float)
    if len(actual) == 0 or actual.sum() <= 0:
        raise ValueError("没有可评估的未缺货销量")
    assert np.isfinite(actual).all() and np.isfinite(predicted).all()
    return {
        **demand_metrics(actual, predicted),
        "mae": np.abs(predicted - actual).mean(),
        "rmse": np.sqrt(np.mean((predicted - actual) ** 2)),
        "evaluated_days": len(actual),
    }

### 6.2 滚动起点回测

初始历史 730 天，每 28 天重新发起一次 28 天预测。本教学数据假设折扣计划提前已知；真实系统若保存的是事后实际价格，内置回测不会替你还原历史计划快照。

下面计算 12 个起点，单机顺序执行。结果中的 `coverage` 是观测覆盖率，不是库存服务水平；默认区间也没有自动涵盖促销计划变动风险。

In [ ]:
from prophet.diagnostics import cross_validation, performance_metrics

cv = cross_validation(
    model, initial="730 days", period="28 days", horizon="28 days", parallel=None
)
scores = performance_metrics(
    cv, rolling_window=0.1
)  # 10% 样本滚动汇总，不是精确步长分组。
display(scores[["horizon", "mae", "rmse", "coverage"]].tail())
assert cv["cutoff"].nunique() == 12
cv_exact = cv.assign(horizon_days=(cv["ds"] - cv["cutoff"]).dt.days)
exact_horizon_metrics = cv_exact.groupby("horizon_days").apply(
    lambda g: pd.Series(
        {
            "n": len(g),
            "mae": np.abs(g["y"] - g["yhat"]).mean(),
            "coverage": g["y"].between(g["yhat_lower"], g["yhat_upper"]).mean(),
        }
    ),
    include_groups=False,
)
display(exact_horizon_metrics)
exact_horizon_metrics.to_csv(OUTPUT_DIR / "exact_horizon_metrics.csv")

### 6.3 参数对照实验

以下都是教学实验，不替换 `model`。主案例的参数对照只使用最后测试期之前的两个内部 28 天窗口；第 16 节才用它们选择配置并报告最终测试结果。不同候选位置范围和先验尺度回答不同的问题。


In [ ]:
def teaching_model(
    cps=0.05,
    cp_range=0.80,
    yearly_order=5,
    season_prior=1.0,
    omit=(),
    mode="additive",
    main_calendar=True,
):
    fitted = Prophet(
        growth="linear",
        n_changepoints=25,
        changepoint_range=cp_range,
        changepoint_prior_scale=cps,
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode=mode,
        seasonality_prior_scale=season_prior,
        holidays=vic_holidays if main_calendar and "holidays" not in omit else None,
        holidays_mode="additive",
        holidays_prior_scale=1.0,
        mcmc_samples=0,
        uncertainty_samples=0,
    )
    fitted.add_seasonality("weekly", period=7, fourier_order=3)
    if yearly_order and "yearly" not in omit:
        fitted.add_seasonality("yearly", period=365.25, fourier_order=yearly_order)
    if "discount" not in omit:
        fitted.add_regressor(
            "discount", prior_scale=1.0, standardize=False, mode="additive"
        )
    return fitted


def fit_teaching(history, config):
    fitted = teaching_model(**config)
    columns = ["ds", "y"] + (
        ["discount"] if "discount" in fitted.extra_regressors else []
    )
    fitted.fit(history[columns], seed=42)
    return fitted


def predict_teaching(fitted, frame):
    columns = ["ds"] + (["discount"] if "discount" in fitted.extra_regressors else [])
    return fitted.predict(frame[columns])


internal_origins = [cutoff - pd.Timedelta(days=56), cutoff - pd.Timedelta(days=28)]


def validate_config(config):
    folds = []
    training_maes = []
    for origin in internal_origins:
        history = train.loc[train["ds"] <= origin].copy()
        validation = train.loc[
            train["ds"].gt(origin) & train["ds"].le(origin + pd.Timedelta(days=28))
        ].copy()
        assert validation["ds"].max() <= cutoff
        fitted = fit_teaching(history, config)
        predicted = predict_teaching(fitted, validation)
        frame = validation.copy()
        frame["forecast_units"] = sales_from_open_forecast(
            validation, predicted["yhat"]
        )
        frame["cutoff"] = origin
        folds.append(frame)
        valid_history = history.dropna(subset=["y"])
        fit_values = predict_teaching(fitted, valid_history)["yhat"].to_numpy()
        training_maes.append(np.abs(valid_history["y"].to_numpy() - fit_values).mean())
    combined = pd.concat(folds, ignore_index=True)
    return {
        "train_mae": np.mean(training_maes),
        **comparison_metrics(combined, "forecast_units"),
    }, combined

### 趋势灵活度与近期转折

独立生成 700 天数据，第 600 天之后斜率由 0.025 增加到 0.275，最后 28 天作为此实验的验证期。转折位于训练历史末段，80% 范围没有合适的候选位置。只提高先验尺度未必能修复候选位置不足；95% 范围则覆盖转折附近。


In [ ]:
trend_rng = np.random.default_rng(314)
trend_days = np.arange(700)
trend_truth = 100 + 0.025 * trend_days + 0.25 * np.maximum(trend_days - 600, 0)
trend_data = pd.DataFrame(
    {
        "ds": pd.date_range("2023-01-01", periods=700),
        "y": trend_truth + trend_rng.normal(0, 2, 700),
        "discount": 0.0,
    }
)
trend_history, trend_validation = (
    trend_data.iloc[:-28].copy(),
    trend_data.iloc[-28:].copy(),
)
trend_rows = []
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(trend_data["ds"], trend_truth, color="black", label="Known trend")
for cp_range, cps in [(0.8, 0.001), (0.8, 0.05), (0.8, 0.5), (0.95, 0.05), (0.95, 0.5)]:
    cfg = dict(
        cp_range=cp_range,
        cps=cps,
        yearly_order=0,
        omit=("discount", "holidays"),
        main_calendar=False,
    )
    fitted = fit_teaching(trend_history, cfg)
    prediction = predict_teaching(fitted, trend_data)
    slope = (
        (fitted.params["k"].mean() + fitted.params["delta"].mean(axis=0).sum())
        * fitted.y_scale
        / (fitted.t_scale / pd.Timedelta(days=1))
    )
    trend_rows.append(
        {
            "range": cp_range,
            "cps": cps,
            "last_candidate": str(fitted.changepoints.max().date()),
            "train_mae": np.abs(
                trend_history["y"].to_numpy() - prediction["yhat"].iloc[:-28].to_numpy()
            ).mean(),
            "validation_mae": np.abs(
                trend_validation["y"].to_numpy()
                - prediction["yhat"].iloc[-28:].to_numpy()
            ).mean(),
            "final_slope": slope,
        }
    )
    ax.plot(prediction["ds"], prediction["trend"], label=f"range={cp_range}, cps={cps}")
ax.axvline(
    trend_data["ds"].iloc[600], color="gray", linestyle=":", label="True changepoint"
)
ax.axvline(trend_history["ds"].max(), color="red", linestyle="--", label="Origin")
ax.set_xlim(trend_data["ds"].iloc[500], trend_data["ds"].max())
ax.set(
    ylabel="Units per day", title="Recent change: candidate location versus shrinkage"
)
ax.legend(fontsize=8)
save_plot(fig, OUTPUT_DIR / "trend_sensitivity.png")
plt.show()
trend_experiment = pd.DataFrame(trend_rows)
display(trend_experiment)
trend_experiment.to_csv(OUTPUT_DIR / "trend_sensitivity.csv", index=False)

### 年周期阶数与先验尺度

阶数决定可表达的曲线空间，先验尺度决定系数收缩。下面同时报告内部训练 MAE 和两个内部 28 天窗口的验证指标；训练拟合更好不意味着未来误差更低。


In [ ]:
season_rows = []
season_configs = [
    dict(yearly_order=1, season_prior=0.01),
    dict(yearly_order=5, season_prior=0.01),
    dict(yearly_order=5, season_prior=1.0),
    dict(yearly_order=15, season_prior=1.0),
]
fig, ax = plt.subplots(figsize=(11, 4))
season_probe = pd.DataFrame(
    {"ds": pd.date_range("2025-01-01", periods=365), "discount": 0.0}
)
for cfg in season_configs:
    metrics, _ = validate_config(cfg)
    season_rows.append({**cfg, **metrics})
    fitted = fit_teaching(train, cfg)
    curve = predict_teaching(fitted, season_probe)
    ax.plot(
        curve["ds"],
        curve["yearly"],
        label=f"order={cfg['yearly_order']}, prior={cfg['season_prior']}",
    )
ax.set(
    title="Yearly complexity versus coefficient shrinkage",
    ylabel="Contribution (units)",
)
ax.legend()
save_plot(fig, OUTPUT_DIR / "seasonality_sensitivity.png")
plt.show()
season_experiment = pd.DataFrame(season_rows)
display(season_experiment)
season_experiment.to_csv(OUTPUT_DIR / "seasonality_sensitivity.csv", index=False)

### 模块消融：遗漏效应被谁吸收？

逐一移除折扣、假日或年周期，在相同内部起点比较，并展示对应日期切片。切片可重叠：一个假日也可能促销，不能把切片样本数相加当作总数。


In [ ]:
ablation_rows = []
ablation_slices = []
ablation_component_rows = []
for name, omit in [
    ("full", ()),
    ("no_discount", ("discount",)),
    ("no_holidays", ("holidays",)),
    ("no_yearly", ("yearly",)),
]:
    metrics, folds = validate_config(dict(omit=omit))
    ablation_rows.append({"model": name, **metrics})
    for slice_name, mask in [
        ("promotion", folds["discount"].gt(0)),
        ("holiday", folds["ds"].isin(vic_holidays["ds"])),
        ("ordinary", folds["discount"].eq(0) & ~folds["ds"].isin(vic_holidays["ds"])),
    ]:
        subset = folds.loc[mask]
        if (~subset["stockout"]).any():
            ablation_slices.append(
                {
                    "model": name,
                    "slice": slice_name,
                    **comparison_metrics(subset, "forecast_units"),
                }
            )
    fitted = fit_teaching(train, dict(omit=omit))
    probe_prediction = predict_teaching(fitted, probe_dates)
    row = {"model": name}
    for col in ["trend", "weekly", "yearly", "holidays", "extra_regressors_additive"]:
        row[col] = (
            float(probe_prediction[col].iloc[1]) if col in probe_prediction else 0.0
        )
    ablation_component_rows.append(row)
ablation_experiment = pd.DataFrame(ablation_rows)
display(ablation_experiment)
display(pd.DataFrame(ablation_slices))
display(pd.DataFrame(ablation_component_rows))
ablation_experiment.to_csv(OUTPUT_DIR / "ablation_metrics.csv", index=False)
pd.DataFrame(ablation_slices).to_csv(OUTPUT_DIR / "ablation_slices.csv", index=False)
pd.DataFrame(ablation_component_rows).to_csv(
    OUTPUT_DIR / "ablation_components.csv", index=False
)

### 加法与乘法：单独生成两种机制

两组数据使用相同增长水平与零均值周模式。加法组周波动幅度固定；乘法组周波动随基础水平增大。它们是独立实验，不修改主案例。比较两种拟合模式在末段 28 天对无噪声条件均值的误差。


In [ ]:
mode_rng = np.random.default_rng(2718)
mode_dates = pd.date_range("2023-01-01", periods=730)
level = 50 + 0.15 * np.arange(730)
centered_week = (mode_dates.dayofweek >= 5).astype(float) - 2 / 7
mode_rows = []
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for axis, (mechanism, truth) in zip(
    axes,
    [
        ("fixed_units", level + 14 * centered_week),
        ("proportional", level * (1 + 0.28 * centered_week)),
    ],
):
    frame = pd.DataFrame(
        {"ds": mode_dates, "y": truth + mode_rng.normal(0, 2, len(mode_dates))}
    )
    history, validation = frame.iloc[:-28].copy(), frame.iloc[-28:].copy()
    axis.plot(
        validation["ds"], truth[-28:], color="black", label="Known conditional mean"
    )
    for mode in ["additive", "multiplicative"]:
        cfg = dict(
            mode=mode,
            yearly_order=0,
            omit=("discount", "holidays"),
            main_calendar=False,
        )
        fitted = fit_teaching(history, cfg)
        prediction = predict_teaching(fitted, validation)
        mode_rows.append(
            {
                "mechanism": mechanism,
                "mode": mode,
                "mean_mae": np.abs(prediction["yhat"].to_numpy() - truth[-28:]).mean(),
                "observed_mae": np.abs(
                    prediction["yhat"].to_numpy() - validation["y"].to_numpy()
                ).mean(),
            }
        )
        axis.plot(prediction["ds"], prediction["yhat"], label=mode)
    axis.set(title=mechanism, ylabel="Units")
    axis.legend()
save_plot(fig, OUTPUT_DIR / "additive_multiplicative.png")
plt.show()
mode_experiment = pd.DataFrame(mode_rows)
display(mode_experiment)
mode_experiment.to_csv(OUTPUT_DIR / "mode_experiment.csv", index=False)

### 6.4 在训练历史内选择配置

候选在运行前固定，只用第 15 节两个历史起点汇总 WAPE 选择。它是流程演示，不是充分调参。最后窗口已在教程中公开，其结果不能用于事后反复选择；真实项目应另保留未看过的留出期。


In [ ]:
prophet_candidates = [
    dict(cps=0.01, yearly_order=5, season_prior=1.0),
    dict(cps=0.05, yearly_order=5, season_prior=1.0),
    dict(cps=0.05, yearly_order=1, season_prior=0.1),
]
validation_rows = []
for candidate_id, cfg in enumerate(prophet_candidates):
    metrics, _ = validate_config(cfg)
    validation_rows.append({"candidate_id": candidate_id, **cfg, **metrics})
prophet_selection = pd.DataFrame(validation_rows)
best_id = int(
    prophet_selection.sort_values(["wape", "candidate_id"]).iloc[0]["candidate_id"]
)
selected_prophet = fit_teaching(train, prophet_candidates[best_id])
selected_test = result.copy()
selected_test["forecast_units"] = sales_from_open_forecast(
    test, predict_teaching(selected_prophet, test)["yhat"]
)
selected_metrics = comparison_metrics(selected_test, "forecast_units")
display(prophet_selection)
print("Selected config:", prophet_candidates[best_id])
print("Final test:", selected_metrics)
prophet_selection.to_csv(OUTPUT_DIR / "prophet_parameter_selection.csv", index=False)

### 6.5 预测区间与实际覆盖率

保持已拟合 MAP 参数不变，仅使用预测模拟生成 80% 和 95% 区间。共同随机种子便于比较；区间不确定性不会自动改善点预测。12 个离线起点都沿用原先模型参数，按未缺货观测计算覆盖率与平均宽度。


In [ ]:
from prophet.diagnostics import prophet_copy

interval_rows = []
interval_predictions = []
for origin in sorted(cv["cutoff"].unique()):
    origin = pd.Timestamp(origin)
    historical = model.history.loc[model.history["ds"] <= origin].copy()
    interval_model = prophet_copy(model, cutoff=origin)
    interval_model.fit(historical, seed=42)
    inputs = data.loc[
        data["ds"].gt(origin) & data["ds"].le(origin + pd.Timedelta(days=28))
    ].copy()
    point_reference = None
    for width in [0.80, 0.95]:
        interval_model.interval_width = width
        interval_model.uncertainty_samples = 1000
        np.random.seed(42)
        prediction = interval_model.predict(inputs[["ds", "discount"]])
        if point_reference is None:
            point_reference = prediction["yhat"].to_numpy()
        else:
            np.testing.assert_allclose(prediction["yhat"], point_reference)
        scored = inputs.loc[~inputs["stockout"] & inputs["is_open"]]
        mask = (~inputs["stockout"] & inputs["is_open"]).to_numpy()
        evaluated = prediction.loc[mask]
        for actual, lo, hi in zip(
            scored["y"], evaluated["yhat_lower"], evaluated["yhat_upper"]
        ):
            interval_rows.append(
                {
                    "nominal": width,
                    "covered": lo <= actual <= hi,
                    "width": hi - lo,
                    "cutoff": origin,
                }
            )
        prediction["nominal"] = width
        prediction["cutoff"] = origin
        interval_predictions.append(prediction)
interval_detail = pd.DataFrame(interval_rows)
interval_summary = (
    interval_detail.groupby("nominal")
    .agg(
        coverage=("covered", "mean"),
        mean_width=("width", "mean"),
        n=("covered", "size"),
    )
    .reset_index()
)
display(interval_summary)
interval_summary.to_csv(OUTPUT_DIR / "interval_summary.csv", index=False)
last_origin = max(pd.Timestamp(x) for x in cv["cutoff"].unique())
fig, ax = plt.subplots(figsize=(11, 4))
for width, color in [(0.95, "lightblue"), (0.80, "steelblue")]:
    prediction = next(
        p
        for p in interval_predictions
        if p["nominal"].iloc[0] == width and p["cutoff"].iloc[0] == last_origin
    )
    ax.fill_between(
        prediction["ds"],
        prediction["yhat_lower"],
        prediction["yhat_upper"],
        alpha=0.4,
        color=color,
        label=f"{width:.0%} interval",
    )
ax.plot(
    prediction["ds"], prediction["yhat"], color="black", label="Same point forecast"
)
actual = data.loc[
    data["ds"].isin(prediction["ds"]) & ~data["stockout"] & data["is_open"]
]
ax.scatter(
    actual["ds"], actual["y"], s=12, color="red", label="Unrestricted observed sales"
)
fig.autofmt_xdate()
ax.set(title="MAP intervals at last rolling origin", ylabel="Units")
ax.legend()
save_plot(fig, OUTPUT_DIR / "interval_comparison.png")
plt.show()

### 6.6 已知真相与残差诊断

合成数据允许比较无噪声均值、含噪声潜在需求和最终观测销售。最后测试期可能没有缺货，所以销售与潜在需求的区别还需结合历史及第 6.7 节观察。残差图只用于发现遗漏结构，不代替未来窗口评估；删除缺货日期后不能按行号把不连续日期当成连续滞后。


In [ ]:
prophet_truth_rows = []
for target in ["conditional_mean", "latent_demand", "y"]:
    scored = result.loc[result.is_open & ~result.stockout]
    actual = scored[target].to_numpy()
    predicted = scored.forecast_units.to_numpy()
    prophet_truth_rows.append(
        {
            "target": target,
            "model": "Prophet",
            **demand_metrics(actual, predicted),
            "mae": np.abs(actual - predicted).mean(),
        }
    )
prophet_truth_metrics = pd.DataFrame(prophet_truth_rows)
display(prophet_truth_metrics)
prophet_truth_metrics.to_csv(
    OUTPUT_DIR / "prophet_known_truth_metrics.csv", index=False
)

In [ ]:
valid_history = train.dropna(subset=["y"]).copy()
historical_fit = model.predict(valid_history[["ds", "discount"]])
residual = pd.Series(
    valid_history["y"].to_numpy() - historical_fit["yhat"].to_numpy(),
    index=pd.DatetimeIndex(valid_history["ds"]),
)
daily_residual = residual.reindex(
    pd.date_range(train["ds"].min(), train["ds"].max(), freq="D")
)
acf_rows = []
for lag in range(1, 29):
    pairs = pd.concat([daily_residual, daily_residual.shift(lag)], axis=1).dropna()
    acf_rows.append(
        {
            "lag_days": lag,
            "correlation": pairs.iloc[:, 0].corr(pairs.iloc[:, 1]),
            "pairs": len(pairs),
        }
    )
residual_acf = pd.DataFrame(acf_rows)
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
axes[0].plot(daily_residual.index, daily_residual, linewidth=0.7)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(
    title="Training residuals: unavailable labels remain missing",
    ylabel="Observed - fitted",
)
axes[1].stem(residual_acf["lag_days"], residual_acf["correlation"])
axes[1].set(
    xlabel="Calendar lag (days)",
    ylabel="Pairwise correlation",
    title="Residual correlation: exploratory, not a significance test",
)
save_plot(fig, OUTPUT_DIR / "residual_diagnostics.png")
plt.show()
residual_acf.to_csv(OUTPUT_DIR / "residual_acf.csv", index=False)

### 6.7 可选实验：需求相关缺货

不修改主案例。独立生成具有随机正向冲击的需求，固定每日供给 115 件；需求超过供给才缺货。对照直接拟合截断销售与屏蔽缺货标签，评价对象均为实验的潜在需求。它展示标签选择偏差，不能验证真实需求修复算法。


In [ ]:
supply_rng = np.random.default_rng(808)
supply_ds = pd.date_range("2023-01-01", periods=730)
supply_day = np.arange(730)
supply_mean = 90 + 0.02 * supply_day + 18 * (supply_ds.dayofweek >= 5).astype(float)
supply_noise = (
    supply_rng.normal(0, 5, len(supply_ds))
    + supply_rng.exponential(8, len(supply_ds))
    - 8
)
supply_demand = np.maximum(0, supply_mean + supply_noise)
capacity = 115.0
supply_frame = pd.DataFrame(
    {
        "ds": supply_ds,
        "y": np.minimum(supply_demand, capacity),
        "latent_demand": supply_demand,
        "stockout": supply_demand > capacity,
    }
)
supply_history = supply_frame.iloc[:-28].copy()
supply_test = supply_frame.iloc[-28:].copy()
supply_rows = []
for treatment in ["censored_sales", "mask_stockout"]:
    history = supply_history.copy()
    if treatment == "mask_stockout":
        history.loc[history["stockout"], "y"] = np.nan
    fitted = fit_teaching(
        history,
        dict(yearly_order=0, omit=("discount", "holidays"), main_calendar=False),
    )
    prediction = predict_teaching(fitted, supply_test)
    supply_rows.append(
        {
            "treatment": treatment,
            "training_labels": history["y"].notna().sum(),
            **demand_metrics(supply_test["latent_demand"], prediction["yhat"]),
        }
    )
supply_experiment = pd.DataFrame(supply_rows)
display(supply_experiment)
display(
    supply_history.groupby("stockout").agg(
        n=("ds", "size"),
        mean_latent=("latent_demand", "mean"),
        mean_sales=("y", "mean"),
    )
)
supply_experiment.to_csv(OUTPUT_DIR / "demand_dependent_stockout.csv", index=False)

## 七、MLOps：从模型到需求预测系统

生产流程需要连接数据准备、训练与回测、版本发布、预测输出、补货集成和运行监控。本章演示模型保存、点预测恢复与带版本的输出；批量任务调度、真实告警与生产 CI/CT/CD 由工程平台实现。

持续集成（Continuous Integration，CI）检查代码、数据约定和接口；持续训练（Continuous Training，CT）训练并评价候选；持续交付（Continuous Delivery，CD）发布通过质量门槛的版本并支持回滚。持续评估为 CT 提供信号，候选发布还需要数据质量和性能门槛。

### 7.1 保存模型与业务结果

JSON 序列化后验证点预测一致。随机生成的区间端点可能不同，不应把它们当作确定性一致性检查。

预测结果保留原始 `yhat` 和业务非负字段。此处不执行订货量计算：箱规、营业安排、交期与库存策略需要业务规则处理。

In [ ]:
from prophet.serialize import model_to_json, model_from_json

model_file = OUTPUT_DIR / "retail_prophet_vic.json"
model_file.write_text(model_to_json(model), encoding="utf-8")
restored = model_from_json(model_file.read_text(encoding="utf-8"))
restored_forecast = restored.predict(future)
np.testing.assert_allclose(forecast["yhat"], restored_forecast["yhat"])
result.to_csv(OUTPUT_DIR / "retail_forecast_28days.csv", index=False)
print("Saved:", model_file, OUTPUT_DIR / "retail_forecast_28days.csv")

In [ ]:
import hashlib
import json

RUN_ID = f"teaching-{cutoff:%Y%m%d}-prophet140"
manifest = {
    "run_id": RUN_ID,
    "forecast_origin": str(cutoff.date()),
    "versions": {
        name: version(name)
        for name in ["prophet", "numpy", "pandas", "holidays", "lightgbm"]
    },
    "input_sha256": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in required_paths
    },
    "regressors": ["discount"],
    "training_labels": int(train.y.notna().sum()),
    "model_file": model_file.name,
    "holdout_metrics": demand_metrics(observed.y, observed.forecast_units),
    "scope": "synthetic tutorial; production scheduling and deployment not implemented",
}
(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2)
)
delivery = result[["ds", "yhat", "forecast_units"]].copy()
delivery = delivery.rename(columns={"yhat": "yhat_raw"})
delivery["forecast_origin"] = cutoff
delivery["run_id"] = RUN_ID
delivery["model_version"] = "prophet140-main"
delivery["plan_version"] = manifest["input_sha256"]["grocery_scenario_data.csv"]
delivery["status"] = "model"
delivery["fallback_reason"] = ""
assert not delivery.duplicated(["forecast_origin", "ds", "model_version"]).any()
delivery.to_csv(OUTPUT_DIR / "versioned_forecast_28days.csv", index=False)
print("已保存模型、输入校验和与带版本预测；门店/SKU 字段需按实际系统补充。")

### 7.2 接入真实数据与运行交接

准备一个门店商品组合的日级历史 CSV：`ds,y,discount`。日期按门店当地营业日生成、无时区；每日期一行。缺货标签可保留为缺失，但额外变量必须另行处理，不能包含无效值。

准备预测起点当时可见的未来计划 CSV：`ds,discount`，覆盖全部预测日期，不需要 `y`。实际应用需要重新生成适用年份的 VIC 日历，再用新建模型拟合。

这个检查函数只检查接口约定，不能替代业务标签与缺货审计。不要把多门店商品组合混进一个局部模型，也不要直接把丢失日期补为零。

新主案例还需要 `is_open` 营业计划。保留原始闭店零销量，拟合营业需求前屏蔽闭店标签；预测销售时应用同一计划。下面 CSV 读取函数只演示数值特征与日期校验，营业日历需按门店 ID 和日期另行合并，不能把缺少营业状态默认为已经正确处理闭店。

CSV 校验函数 `read_retail_csv()` 已在第 2.1 节定义并用于实际加载。替换真实数据时，接手团队还应核对门店日历、库存、在售范围、未来计划版本与回退条件；保存并复现一个运行版本，再验证生产接口。

## 八、模型拓展与其他模型比较

Prophet 显式指定趋势、周期与业务项；ARIMA 结合差分、自回归和误差结构，带 ARIMA 误差的回归可加入业务变量；LightGBM 与 XGBoost 用提升树学习阈值与交互；TFT 区分静态、已知未来及历史输入，适用于具体的多步预测设计。基本原理及参考文献见博客第八章。

这里实际运行多业务特征 Prophet、LightGBM 和基线。ARIMA、XGBoost、TFT 不提供未运行的误差数字。所有比较需使用相同起点可见信息和标签口径。

### 8.1 扩展多业务特征

下方函数使用价格、陈列和广告投入。训练和预测都需要对应列，未来输入必须在发起预测时可用。单条序列中固定不变的门店面积、品牌等，通常应放入分组策略或全局模型。

价格与折扣可能存在确定关系，不能盲目同时加入。未知天气要使用外部预测或情景值；销量滞后不能读取未来实际值。本函数没有自动增加销量滞后或残差自回归。

In [ ]:
def fit_with_business_features(history, forecast_inputs, holiday_calendar):
    # history: ds、y 和下面三列历史业务特征。
    # forecast_inputs: ds 和未来目标日的计划特征；不需要 y。
    # 两张表的时间、单位和特征编码必须保持一致。
    feature_settings = {
        "unit_price": True,  # 商品单位售价，单位为 AUD。
        "display": False,  # 是否参与指定陈列活动，编码为 0/1。
        "ad_spend": True,  # 按明确业务口径归属到该序列的广告投入。
    }
    feature_cols = list(feature_settings)

    model = Prophet(
        holidays=holiday_calendar,
        growth="linear",
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode="additive",
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=1.0,
        holidays_prior_scale=1.0,
    )
    model.add_seasonality("weekly", period=7, fourier_order=3)
    model.add_seasonality("yearly", period=365.25, fourier_order=5)

    for name, standardize in feature_settings.items():
        model.add_regressor(
            name,
            standardize=standardize,
            prior_scale=1.0,
            mode="additive",
        )

    model.fit(history[["ds", "y", *feature_cols]], seed=42)
    prediction = model.predict(forecast_inputs[["ds", *feature_cols]])
    return model, prediction

下面单独生成三特征的教学样本，让目标确实依赖价格、陈列和广告。所有未来安排都是合成的预先已知计划。该单元用于理解接口，不用于报告真实业务收益，也不替换主案例的验证结果。

In [ ]:
extra_rng = np.random.default_rng(2026)
business_data = pd.DataFrame({"ds": ds})
business_data["unit_price"] = 10 - 2 * discount
business_data["display"] = (t % 17 < 4).astype(int)
business_data["ad_spend"] = np.where(t % 23 < 6, 40.0, 0.0)
business_data["y"] = np.rint(
    np.maximum(
        0,
        120
        + 0.02 * t
        + 18 * weekend
        + annual
        + holiday_uplift
        - 4 * business_data["unit_price"].to_numpy()
        + 8 * business_data["display"].to_numpy()
        + 0.15 * business_data["ad_spend"].to_numpy()
        + extra_rng.normal(0, 5, len(ds)),
    )
)
extra_history = business_data.loc[business_data["ds"] <= cutoff].copy()
extra_future = business_data.loc[business_data["ds"] > cutoff].drop(columns="y").copy()
extra_model, extra_prediction = fit_with_business_features(
    extra_history, extra_future, vic_holidays
)
display(extra_prediction[["ds", "yhat", "extra_regressors_additive"]].head())
display(regressor_coefficients(extra_model))
assert len(extra_prediction) == 28
assert np.isfinite(extra_prediction["yhat"]).all()
price_coefficient = (
    regressor_coefficients(extra_model).set_index("regressor").loc["unit_price", "coef"]
)
price_center = extra_model.extra_regressors["unit_price"]["mu"]
prepared_extra = extra_model.setup_dataframe(
    extra_future.copy(), initialize_scales=False
)
XF, _, CM, _ = extra_model.make_all_seasonality_features(prepared_extra)
price_indices = CM["unit_price"].to_numpy().astype(bool)
price_from_internal = (
    XF.to_numpy()[:, price_indices]
    @ extra_model.params["beta"].mean(axis=0)[price_indices]
) * extra_model.y_scale
price_from_original = price_coefficient * (
    extra_future["unit_price"].to_numpy() - price_center
)
np.testing.assert_allclose(price_from_original, price_from_internal)
print("Price training center:", price_center)

### 8.2 同数据的 LightGBM 与 Prophet

LightGBM 是梯度提升决策树（Gradient Boosting Decision Tree, GBDT）的实现。它依次增加树，沿损失函数的负梯度方向改善拟合；这种训练迭代与“预测明天，再把明天喂回模型”是两件事。本节使用 `LGBMRegressor`，一次生成测试期各日期的回归预测。

我们使用主案例的 `data`、`train`、`test` 和 `vic_holidays`，不使用第 8.1 节另行生成的 `business_data`。两类模型得到相同的日期、VIC 假日和已知折扣计划；LightGBM 不自动理解 `ds`，需要我们构造数值特征。

| 输入 | LightGBM 特征 | 零售含义 |
|---|---|---|
| `ds` | 距固定起点的天数 | 表达基础销量随时间变化；树模型的这种特征不保证能外推线性增长 |
| `ds` | 星期几及周周期正弦、余弦 | 工作日、周末重复规律；周周期阶数与 Prophet 一致 |
| `ds` | 年周期正弦、余弦 | 年内重复需求波动；年周期阶数与 Prophet 一致 |
| VIC 日历 | 按假日名称生成 0/1 列 | 每种假日的当天影响，与主案例窗口一致 |
| `discount` | 原始折扣比例 | 预测起点已知的未来促销计划 |

两类模型都排除缺货日训练标签，测试指标只使用同一组未缺货日，并将点预测截断到非负值。`stockout` 仅用于标签处理和评估筛选，不作为未来特征；未来 `y` 不进入特征构造。

本对照暂不加入昨日销量或过去七天均值，以保持可用信息一致。如果以后加入这些特征，需要重新明确 28 天预测策略：递归预测只能更新预测值；直接多步模型只能使用各预测起点之前的历史。不能通过对完整数据 `shift(1)` 就声称未来 28 天没有泄漏。


In [ ]:
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

print("LightGBM:", version("lightgbm"))

# 固定日历定义，不根据销量或测试误差选择特征。
TIME_EPOCH = pd.Timestamp("1970-01-01")
holiday_names = sorted(vic_holidays["holiday"].unique())
holiday_dates = {
    name: pd.DatetimeIndex(vic_holidays.loc[vic_holidays["holiday"].eq(name), "ds"])
    for name in holiday_names
}


def make_lgbm_features(frame):
    # 有意只读取日期和折扣；即使 frame 带有 y，也不会读取该列。
    dates = pd.to_datetime(frame["ds"])
    days = (dates - TIME_EPOCH) / pd.Timedelta(days=1)
    X = pd.DataFrame(index=frame.index)
    X["days_since_2023"] = (dates - pd.Timestamp("2023-01-01")) / pd.Timedelta(days=1)
    X["weekday"] = dates.dt.dayofweek
    for period_name, period, order in [("weekly", 7, 3), ("yearly", 365.25, 5)]:
        for harmonic in range(1, order + 1):
            angle = 2 * np.pi * harmonic * days / period
            X[f"{period_name}_sin_{harmonic}"] = np.sin(angle)
            X[f"{period_name}_cos_{harmonic}"] = np.cos(angle)
    for number, name in enumerate(holiday_names):
        # 使用简单列名，避免假日名称中特殊字符影响模型接口。
        X[f"holiday_{number:02d}"] = dates.isin(holiday_dates[name]).astype(int)
    X["discount"] = frame["discount"].astype(float)
    assert "y" not in X and "stockout" not in X
    assert np.isfinite(X.to_numpy()).all()
    return X


display(make_lgbm_features(train).head())
display(
    pd.DataFrame(
        {
            "feature": [f"holiday_{i:02d}" for i in range(len(holiday_names))],
            "holiday": holiday_names,
        }
    )
)

### 用训练期内部的时间验证选择树的数量

每次拟合都从当前训练历史末尾留出 56 个日历日，使用更早的有效标签训练临时模型，以验证集 MAE 进行早停（early stopping）。随后用选出的树数量在全部有效训练历史上重新拟合。最终 28 天测试标签不参与早停或调参。

`num_leaves=15` 限制单棵树复杂度；`min_child_samples=20` 降低少量样本支撑过细规则的风险；`learning_rate=0.03` 控制每轮更新幅度；`reg_lambda=1.0` 提供叶权重正则化。最多 800 棵树只是早停搜索上限，`n_jobs=2` 限制单模型线程数，便于以后安排多序列并行训练。这些是教学配置，真实业务需要在历史起点上验证。

两类模型使用相同数据和评估协议，但本例没有给两类模型做等预算的参数搜索，结果只能说明当前配置的表现。标准回归输出也不提供与 Prophet 预测区间对应的区间，不能据此比较区间覆盖率。


In [ ]:
LGBM_SETTINGS = dict(
    objective="regression",  # 训练时最小化平方误差，验证时用 MAE 选迭代数。
    metric="mae",  # 显式覆盖默认验证指标，使早停只关注 MAE。
    learning_rate=0.03,  # 每轮树更新幅度。
    num_leaves=15,  # 单棵树最多 15 个叶子，控制复杂度。
    min_child_samples=20,  # 限制少样本叶节点。
    reg_lambda=1.0,  # 叶权重的 L2 正则化。
    random_state=42,  # 固定随机种子。
    n_jobs=2,  # 批量需求预测时控制每个模型的线程数。
    verbosity=-1,
    importance_type="gain",  # 后面展示分裂带来的训练损失改善。
)


def fit_lgbm_at_origin(history):
    history = history.sort_values("ds").copy()
    valid = history.dropna(subset=["y"]).copy()
    # cutoff 在日历上定义，不因删除缺货标签改变日期边界。
    validation_start = history["ds"].max() - pd.Timedelta(days=55)
    inner_train = valid.loc[valid["ds"] < validation_start]
    inner_valid = valid.loc[valid["ds"] >= validation_start]
    if len(inner_train) < 100 or len(inner_valid) < 20:
        raise ValueError("需要足够历史支持 56 天内部时间验证")
    assert inner_train["ds"].max() < inner_valid["ds"].min()
    candidate = LGBMRegressor(n_estimators=800, **LGBM_SETTINGS)
    candidate.fit(
        make_lgbm_features(inner_train),
        inner_train["y"],
        eval_set=[(make_lgbm_features(inner_valid), inner_valid["y"])],
        eval_metric="mae",
        callbacks=[
            early_stopping(50, first_metric_only=True, verbose=False),
            log_evaluation(0),
        ],
    )
    best_iteration = candidate.best_iteration_ or 800
    fitted = LGBMRegressor(n_estimators=best_iteration, **LGBM_SETTINGS)
    fitted.fit(make_lgbm_features(valid), valid["y"])
    return fitted, best_iteration


lgbm_model, selected_trees = fit_lgbm_at_origin(train)
lgbm_inputs = test[["ds", "discount"]].copy()
lgbm_raw = lgbm_model.predict(make_lgbm_features(lgbm_inputs))
comparison = result.copy()
comparison["lightgbm_raw"] = lgbm_raw
comparison["lightgbm_units"] = sales_from_open_forecast(comparison, lgbm_raw)
assert np.isfinite(lgbm_raw).all() and len(lgbm_raw) == 28
assert train["ds"].max() < lgbm_inputs["ds"].min()
print("Training cutoff:", cutoff, "| selected trees:", selected_trees)

prediction_columns = {
    "Prophet": "forecast_units",
    "LightGBM": "lightgbm_units",
    "Seasonal naive": "seasonal_naive",
}
holdout_comparison = pd.DataFrame(
    {
        name: comparison_metrics(comparison, column)
        for name, column in prediction_columns.items()
    }
).T
display(
    holdout_comparison.style.format(
        {
            "wape": "{:.2%}",
            "bias": "{:+.2%}",
            "mae": "{:.2f}",
            "rmse": "{:.2f}",
            "evaluated_days": "{:.0f}",
        }
    )
)
comparison.to_csv(OUTPUT_DIR / "prophet_lightgbm_28days.csv", index=False)
holdout_comparison.to_csv(OUTPUT_DIR / "prophet_lightgbm_holdout_metrics.csv")

WAPE 越低表示累计绝对误差越小；bias 接近零表示总体高估与低估较少。MAE 是平均每天偏差的件数；RMSE 对少数较大的日级误差更敏感。它们应结合营业日、节假日和促销切片理解，不能只看一项指标。

下图显示全部 28 天，缺货日用叉号标出，指标表仍排除这些日期。特征重要性使用训练分裂的 gain；它不是因果贡献，也不等于 Prophet 的销量组件。相关周期特征之间可能分摊重要性。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)
axes[0].plot(
    comparison["ds"], comparison["y"], "o-", color="black", label="Observed sales"
)
for name, column in prediction_columns.items():
    axes[0].plot(comparison["ds"], comparison[column], label=name)
limited = comparison.loc[comparison["stockout"]]
axes[0].scatter(
    limited["ds"],
    limited["y"],
    marker="x",
    color="red",
    s=90,
    label="Stockout: excluded from metrics",
)
axes[0].set(title="Same-origin 28-day holdout", ylabel="Units per day")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=30)

feature_gain = pd.Series(
    lgbm_model.feature_importances_, index=lgbm_model.feature_name_, name="gain"
).sort_values()
feature_gain.tail(12).plot.barh(ax=axes[1])
axes[1].set(title="LightGBM training split gain: not causal effects", xlabel="Gain")
save_plot(fig, OUTPUT_DIR / "prophet_lightgbm_comparison.png")
plt.show()

### 在相同的 12 个滚动起点上核对表现

单个测试窗口可能偏向某类模型。下面复用第 10 节 Prophet 回测的 12 个截止日与预测日期；LightGBM 在每个截止日重新执行内部时间验证和全历史拟合，季节朴素模型也只读取该截止日之前的销量。

所有模型都使用已知的合成折扣计划、同一 VIC 日历及同一缺货筛选。汇总 WAPE 按全部起点的绝对误差总和除以实际销量总和计算，不直接平均每个窗口的百分比。回测用于历史诊断，最后的 28 天测试期保持独立；不要看到测试结果后反复修改配置，再把同一测试集当作未使用的验证证据。


Prophet 内置回测省略缺失训练标签对应的日期，因此我们按完整 28 天窗口对齐结果；缺货日的 Prophet 预测可能为空，这些日期不参与任一模型的指标。


In [ ]:
rolling_frames = []
for origin, prophet_fold in cv.groupby("cutoff", sort=True):
    fold_history = data.loc[data["ds"] <= origin].copy()
    fold_history.loc[fold_history["stockout"] | ~fold_history["is_open"], "y"] = np.nan
    fold_test = data.loc[
        data["ds"].gt(origin) & data["ds"].le(origin + pd.Timedelta(days=28))
    ].copy()
    assert fold_history["ds"].max() < fold_test["ds"].min()
    fold_model, tree_count = fit_lgbm_at_origin(fold_history)
    fold = fold_test.merge(
        prophet_fold[["ds", "yhat"]], on="ds", how="left", validate="one_to_one"
    )
    assert fold.loc[~fold["stockout"] & fold["is_open"], "yhat"].notna().all()
    fold["forecast_units"] = sales_from_open_forecast(fold, fold["yhat"])
    fold["lightgbm_units"] = sales_from_open_forecast(
        fold, fold_model.predict(make_lgbm_features(fold[["ds", "discount"]]))
    )
    weekday_history = fold_history.dropna(subset=["y"]).copy()
    weekday_history["weekday"] = weekday_history["ds"].dt.dayofweek
    weekday_last = weekday_history.groupby("weekday")["y"].last()
    fold["seasonal_naive"] = sales_from_open_forecast(
        fold, fold["ds"].dt.dayofweek.map(weekday_last)
    )
    fold["cutoff"] = origin
    fold["selected_trees"] = tree_count
    assert len(fold) == 28
    rolling_frames.append(fold)

rolling_comparison = pd.concat(rolling_frames, ignore_index=True)
assert rolling_comparison["cutoff"].nunique() == 12
assert rolling_comparison["ds"].max() <= cutoff
assert not rolling_comparison.duplicated(["cutoff", "ds"]).any()
rolling_metrics = pd.DataFrame(
    {
        name: comparison_metrics(rolling_comparison, column)
        for name, column in prediction_columns.items()
    }
).T
display(
    rolling_metrics.style.format(
        {
            "wape": "{:.2%}",
            "bias": "{:+.2%}",
            "mae": "{:.2f}",
            "rmse": "{:.2f}",
            "evaluated_days": "{:.0f}",
        }
    )
)
by_origin = pd.DataFrame(
    [
        {"cutoff": origin, "model": name, **comparison_metrics(fold, column)}
        for origin, fold in rolling_comparison.groupby("cutoff")
        for name, column in prediction_columns.items()
    ]
)
fig, ax = plt.subplots(figsize=(11, 4))
for name, group in by_origin.groupby("model"):
    ax.plot(group["cutoff"], group["wape"] * 100, marker="o", label=name)
ax.set(
    title="Rolling-origin comparison: 28-day forecasts",
    ylabel="WAPE (%)",
    xlabel="Forecast origin",
)
ax.legend()
save_plot(fig, OUTPUT_DIR / "prophet_lightgbm_rolling_wape.png")
plt.show()
rolling_comparison.to_csv(
    OUTPUT_DIR / "prophet_lightgbm_rolling_predictions.csv", index=False
)
rolling_metrics.to_csv(OUTPUT_DIR / "prophet_lightgbm_rolling_metrics.csv")
by_origin.to_csv(OUTPUT_DIR / "prophet_lightgbm_metrics_by_origin.csv", index=False)
print("Holdout metrics (raw fractions):")
print(holdout_comparison.to_string())
print("Rolling metrics (raw fractions):")
print(rolling_metrics.to_string())

这组数据按线性趋势、平滑周期和加法效应生成，结构接近 Prophet 的假设。树模型可以学习非线性关系与特征交互，但仅凭时间索引通常不能延续超出训练范围的增长趋势；少见假日也可能缺少足够样本支持分裂。本实验的排序不能直接用于选择真实业务模型。

如果真实数据有明显价格阈值、陈列与促销交互、多门店商品共同规律，可以继续评估 LightGBM 全局模型，并加入预测起点可用的业务及历史特征。模型选择应结合滚动回测、关键商品切片、维护成本与补货决策表现，而不是只比较一次总体 WAPE。


### 8.3 共同的真实值、促销与假日切片

第六章检查 Prophet 的真实值与残差；现在在已完成 LightGBM 回测后，对三个模型计算相同日期切片与步长指标。

In [ ]:
truth_rows = []
for target in ["conditional_mean", "latent_demand", "y"]:
    for name, column in prediction_columns.items():
        truth_mask = test["is_open"].to_numpy()
        actual = test.loc[truth_mask, target].to_numpy()
        predicted = comparison.loc[truth_mask, column].to_numpy()
        truth_rows.append(
            {
                "target": target,
                "model": name,
                **demand_metrics(actual, predicted),
                "mae": np.abs(actual - predicted).mean(),
            }
        )
truth_metrics = pd.DataFrame(truth_rows)
display(truth_metrics)
truth_metrics.to_csv(OUTPUT_DIR / "known_truth_metrics.csv", index=False)
retail_slices = []
for name, column in prediction_columns.items():
    for slice_name, mask in [
        ("promotion", rolling_comparison["discount"].gt(0)),
        ("holiday", rolling_comparison["ds"].isin(vic_holidays["ds"])),
        (
            "ordinary",
            rolling_comparison["discount"].eq(0)
            & ~rolling_comparison["ds"].isin(vic_holidays["ds"]),
        ),
    ]:
        frame = rolling_comparison.loc[mask]
        retail_slices.append(
            {"model": name, "slice": slice_name, **comparison_metrics(frame, column)}
        )
slice_metrics = pd.DataFrame(retail_slices)
display(slice_metrics)
slice_metrics.to_csv(OUTPUT_DIR / "retail_slice_metrics.csv", index=False)
exact_comparison = []
for horizon, frame in rolling_comparison.assign(
    horizon_days=(rolling_comparison["ds"] - rolling_comparison["cutoff"]).dt.days
).groupby("horizon_days"):
    for name, column in prediction_columns.items():
        exact_comparison.append(
            {
                "horizon_days": horizon,
                "model": name,
                **comparison_metrics(frame, column),
            }
        )
pd.DataFrame(exact_comparison).to_csv(
    OUTPUT_DIR / "model_metrics_by_horizon.csv", index=False
)
display(pd.DataFrame(exact_comparison).head(9))

# 销售业务输出：另按全部未缺货日期评价，明确包含闭店零值。
sales_rows = []
for name, column in prediction_columns.items():
    score = comparison.loc[~comparison["stockout"]]
    a = score["y"].to_numpy()
    p = score[column].to_numpy()
    sales_rows.append(
        {
            "model": name,
            "scope": "all calendar days excluding stockouts",
            **demand_metrics(a, p),
            "mae": np.abs(a - p).mean(),
            "n": len(a),
        }
    )
sales_calendar_metrics = pd.DataFrame(sales_rows)
display(sales_calendar_metrics)
sales_calendar_metrics.to_csv(OUTPUT_DIR / "sales_calendar_metrics.csv", index=False)

### 8.4 稀疏假日特征与训练内候选

先检查原配置的每种假日正例数、split 次数与 gain。随后固定三个候选，在相同训练期内部起点比较：具名假日/最小叶样本 20，具名假日/最小叶样本 2，共享假日标记/最小叶样本 20。共享标记引入“不同假日共享效应”的假设；这里不根据最后测试集成绩选择候选。


In [ ]:
X_valid = make_lgbm_features(train.dropna(subset=["y"]))
holiday_columns = [col for col in X_valid if col.startswith("holiday_")]
holiday_diagnostics = pd.DataFrame(
    {
        "feature": holiday_columns,
        "holiday": holiday_names,
        "positive_days": [int(X_valid[col].sum()) for col in holiday_columns],
        "split_count": [
            int(
                lgbm_model.booster_.feature_importance("split")[
                    lgbm_model.feature_name_.index(col)
                ]
            )
            for col in holiday_columns
        ],
        "gain": [
            float(
                lgbm_model.booster_.feature_importance("gain")[
                    lgbm_model.feature_name_.index(col)
                ]
            )
            for col in holiday_columns
        ],
    }
)
display(holiday_diagnostics)
print("All named holiday gains zero:", holiday_diagnostics["gain"].eq(0).all())
holiday_diagnostics.to_csv(OUTPUT_DIR / "lightgbm_holiday_diagnostics.csv", index=False)


def sparse_features(frame, shared):
    X = make_lgbm_features(frame)
    if shared:
        X = X.drop(columns=holiday_columns)
        X["is_public_holiday"] = frame["ds"].isin(vic_holidays["ds"]).astype(int)
    return X


def fit_sparse_candidate(history, min_leaf, shared):
    valid = history.dropna(subset=["y"]).copy()
    boundary = history["ds"].max() - pd.Timedelta(days=55)
    inner_train = valid.loc[valid["ds"] < boundary]
    inner_valid = valid.loc[valid["ds"] >= boundary]
    settings = {**LGBM_SETTINGS, "min_child_samples": min_leaf}
    candidate = LGBMRegressor(n_estimators=800, **settings)
    candidate.fit(
        sparse_features(inner_train, shared),
        inner_train["y"],
        eval_set=[(sparse_features(inner_valid, shared), inner_valid["y"])],
        eval_metric="mae",
        callbacks=[
            early_stopping(50, first_metric_only=True, verbose=False),
            log_evaluation(0),
        ],
    )
    fitted = LGBMRegressor(n_estimators=candidate.best_iteration_ or 800, **settings)
    fitted.fit(sparse_features(valid, shared), valid["y"])
    return fitted


sparse_candidates = [
    ("named_leaf20", 20, False),
    ("named_leaf2", 2, False),
    ("shared_leaf20", 20, True),
]
sparse_rows = []
for name, min_leaf, shared in sparse_candidates:
    candidate_folds = []
    for origin in internal_origins:
        history = train.loc[train["ds"] <= origin].copy()
        validation = train.loc[
            train["ds"].gt(origin) & train["ds"].le(origin + pd.Timedelta(days=28))
        ].copy()
        fitted = fit_sparse_candidate(history, min_leaf, shared)
        validation["prediction"] = sales_from_open_forecast(
            validation,
            fitted.predict(sparse_features(validation[["ds", "discount"]], shared)),
        )
        candidate_folds.append(validation)
    joined = pd.concat(candidate_folds, ignore_index=True)
    metrics = comparison_metrics(joined, "prediction")
    sparse_rows.append(
        {
            "candidate": name,
            "min_child_samples": min_leaf,
            "shared_holiday": shared,
            **metrics,
        }
    )
sparse_validation = pd.DataFrame(sparse_rows)
display(sparse_validation)
sparse_validation.to_csv(OUTPUT_DIR / "lightgbm_sparse_validation.csv", index=False)
best_sparse_name = sparse_validation.sort_values(["wape", "candidate"]).iloc[0][
    "candidate"
]
_, best_min_leaf, best_shared = next(
    c for c in sparse_candidates if c[0] == best_sparse_name
)
sparse_selected = fit_sparse_candidate(train, best_min_leaf, best_shared)
sparse_final = test.copy()
sparse_final["prediction"] = sales_from_open_forecast(
    test,
    sparse_selected.predict(sparse_features(test[["ds", "discount"]], best_shared)),
)
sparse_final_metrics = comparison_metrics(sparse_final, "prediction")
print("Selected from historical validation:", best_sparse_name)
print("Final test after selection:", sparse_final_metrics)

## 九、总结与实施建议

按顺序运行后，可分别查看第 5.4 节的预测重建、第 6.4 节的历史选参、第 6.5 节的区间校准，以及第 8.2—8.4 节的模型比较。教学数据适合检查计算是否一致；真实业务需要继续验证缺货标签、未来输入、业务分组和运行成本。

下方汇总主要结果与数据来源。先固定问题和评价口径，再决定保留 Prophet、扩展特征或比较其他模型。

In [ ]:
execution_summary = {
    "source": "CSV files generated by prophet-retail-data-generator.ipynb",
    "main_rows": len(data),
    "test_days": len(test),
    "rolling_origins": int(cv.cutoff.nunique()),
    "holdout_wape": holdout_comparison.wape.to_dict(),
    "rolling_wape": rolling_metrics.wape.to_dict(),
    "selected_prophet_config": prophet_candidates[best_id],
    "selected_prophet_test_wape": float(selected_metrics["wape"]),
    "selected_lightgbm_sparse_candidate": best_sparse_name,
    "selected_lightgbm_test_wape": float(sparse_final_metrics["wape"]),
    "interval_coverage": interval_summary.to_dict(orient="records"),
}
(OUTPUT_DIR / "experiment_summary.json").write_text(
    json.dumps(execution_summary, ensure_ascii=False, indent=2)
)
display(execution_summary)

## 附录 A/B：低频红酒与庭院耗材，由读者设计实验

下面读取生成器导出的两个独立练习数据集，不替换主案例，也不宣称已经比较出方案排名。品牌仅帮助理解业务情景；这些消费规则都是教学假设。红酒规格以 750 mL 瓶装为例，销量表示售出瓶数，不是开瓶量。

红酒用正泊松强度采样；Poisson(λ) 自身不等于零膨胀。若要大量零值，可单独改变基线或设计 hurdle/零膨胀机制。12 月冲量、Cup Eve 与跨年夜是业务事件，不应冒充法定公共假日。庭院耗材 clearance 标记为 0/1，系数采用 12，确保单日增加 12 件；不能写 60×1 却解释为 12。

闭店仍使用营业状态门控。为避免错误宣称 Bunnings 只在圣诞节闭店，练习默认把受难周五与圣诞节设为闭店；真实门店必须替换营业日历。不要为防止负预测强行启用 logistic：floor 约束趋势，不保证总预测非负。


In [ ]:
wine_data = read_scenario_csv(
    DATA_DIR / "wine_scenario_data.csv",
    ("promo_case_buy", "is_december_peak", "is_party_event"),
)
garden_data = read_scenario_csv(
    DATA_DIR / "garden_scenario_data.csv",
    ("is_long_weekend_diy", "is_end_of_autumn_clearance"),
)
assert wine_data.ds.equals(data.ds) and garden_data.ds.equals(data.ds)
scenario_diagnostics = []
for name, frame in [("grocery", data), ("wine", wine_data), ("garden", garden_data)]:
    open_y = frame.loc[frame["is_open"], "y"]
    scenario_diagnostics.append(
        {
            "scenario": name,
            "mean_open_sales": open_y.mean(),
            "zero_rate_open": open_y.eq(0).mean(),
            "closed_days": int((~frame["is_open"]).sum()),
            "min_condition_mean_open": frame.loc[
                frame["is_open"], "conditional_mean"
            ].min(),
        }
    )
scenario_diagnostics = pd.DataFrame(scenario_diagnostics)
display(scenario_diagnostics)
scenario_diagnostics.to_csv(OUTPUT_DIR / "scenario_diagnostics.csv", index=False)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for ax, (name, frame) in zip(
    axes, [("Grocery", data), ("Wine", wine_data), ("Garden", garden_data)]
):
    year = frame.loc[frame["ds"].dt.year.eq(2025)]
    ax.plot(year["ds"], year["y"], linewidth=0.7, label="Observed sales")
    ax.plot(
        year["ds"],
        year["conditional_mean"],
        linewidth=1.2,
        label="Noise-free open-day level",
    )
    ax.set(title=name, ylabel="Units")
    ax.legend()
fig.autofmt_xdate()
save_plot(fig, OUTPUT_DIR / "three_scenario_shapes.png")
plt.show()

### 练习接口与研究问题

下面只定义拟合接口，不执行附录模型排名。请在训练历史内部选择假日窗口、年阶数及回归项，再用共同历史起点和相同指标比较模型。评价需求时排除闭店；评价销售时给所有模型应用同一个营业日历。

- 红酒：比较显式事件 Prophet、泊松回归及负二项候选；先测营业日零值比例，再判断是否需要间歇需求方法。检查均值与方差、负预测比例、促销/事件切片。
- 庭院：先用线性趋势与加法季节性，再比较比例波动。验证淡季误差与区间宽度；只有存在可信饱和机制及 cap 时才引入 logistic。
- 跨方案：MAE 的单位规模不同，不能直接用绝对误差排序。WAPE 也随基数、促销频率与相对噪声变化。至少跨多个种子和滚动窗口报告结果。


In [ ]:
def fit_appendix_prophet(frame, origin, name):
    columns = (
        ["promo_case_buy", "is_december_peak", "is_party_event"]
        if name == "wine"
        else ["is_end_of_autumn_clearance", "is_long_weekend_diy"]
    )
    history = frame.loc[frame["ds"] <= pd.Timestamp(origin)].copy()
    history.loc[~history["is_open"] | history["stockout"], "y"] = np.nan
    fitted = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_prior_scale=1.0,
        uncertainty_samples=1000,
    )
    fitted.add_seasonality("weekly", period=7, fourier_order=3)
    fitted.add_seasonality("yearly", period=365.25, fourier_order=5)
    for col in columns:
        fitted.add_regressor(col, standardize=False, prior_scale=1.0)
    # garden 已用 DIY 回归列表达指定事件，不再加相同事件的假日列。
    fitted.fit(history[["ds", "y", *columns]], seed=42)
    future = frame.loc[
        frame["ds"].gt(pd.Timestamp(origin))
        & frame["ds"].le(pd.Timestamp(origin) + pd.Timedelta(days=28))
    ].copy()
    prediction = fitted.predict(future[["ds", *columns]])
    future["raw_demand_yhat"] = prediction["yhat"].to_numpy()
    future["forecast_units"] = sales_from_open_forecast(future, prediction["yhat"])
    return fitted, future


# 由读者执行，并设计训练内验证；此处不生成附录模型性能结论。
# wine_model, wine_result = fit_appendix_prophet(wine_data, cutoff, 'wine')
# garden_model, garden_result = fit_appendix_prophet(garden_data, cutoff, 'garden')

## 参考与实验边界

- [Prophet Quick Start](https://facebook.github.io/prophet/docs/quick_start.html)
- [Seasonality, holidays and regressors](https://facebook.github.io/prophet/docs/seasonality,_holiday_effects,_and_regressors.html)
- [Diagnostics](https://facebook.github.io/prophet/docs/diagnostics.html)
- [Forecast implementation](https://github.com/facebook/prophet/blob/v1.4.0/python/prophet/forecaster.py)
- [Business Victoria: 2025 public holidays](https://business.vic.gov.au/business-information/public-holidays/victorian-public-holidays-2025)

本例数据生成机制与 Prophet 的结构接近，较低误差不代表在真实零售数据上具有相同收益。缺货、计划变更、退市、新品和跨商品替代仍需业务工程处理。
- [LightGBM LGBMRegressor API](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.LGBMRegressor.html)
- [LightGBM early stopping](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.early_stopping.html)

- [Prophet 1.4.0 Stan model](https://github.com/facebook/prophet/blob/v1.4.0/python/stan/prophet.stan)
- [LightGBM 4.6.0 parameters](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html)

- [Business Victoria: restricted trading days](https://business.vic.gov.au/business-information/public-holidays/operating-on-a-restricted-trading-day)
- [Victoria: packaged liquor licence](https://www.vic.gov.au/apply-packaged-liquor-licence)

主案例拟合营业需求，闭店日销售按计划归零。附录只执行 CSV 加载与规则检查，拟合函数留给读者使用，未报告附录模型排名。

## 标准化四模型族比较接口

以下实验与前面的教学诊断分开：它只使用测试期之前的四个滚动窗口选择参数，然后在全部测试前历史上重训并评估最终 28 天。输出格式与 ARIMA、树模型和神经网络一致，供总汇总 Notebook 使用。结果只说明当前数据、特征和有限候选范围的联合表现。

In [ ]:
# 首次使用请在仓库根目录运行：uv sync --group dev
from pathlib import Path
import os, sys, hashlib


def find_repo_root():
    configured = os.environ.get("RETAIL_FORECAST_ROOT")
    candidates = (
        [Path(configured).expanduser()]
        if configured
        else [Path.cwd(), *Path.cwd().parents]
    )
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "notebooks"
        ).is_dir():
            return candidate
    raise RuntimeError(
        "找不到仓库根目录；请从仓库内启动 Jupyter，或设置 RETAIL_FORECAST_ROOT"
    )


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from retail_forecast.common import *
from retail_forecast.prophet_benchmark import *

In [ ]:
benchmark_scenarios = ["grocery", "wine", "garden"]
benchmark = run_prophet_benchmark(benchmark_scenarios, "prophet_retail")
(
    benchmark_metrics,
    benchmark_predictions,
    benchmark_tuning,
    benchmark_parameters,
    benchmark_fits,
    benchmark_explanations,
    benchmark_out,
) = benchmark
display(benchmark_parameters)
display(benchmark_tuning.sort_values(["scenario", "validation_MAE"]).round(4))
display(benchmark_metrics.round(4))

In [ ]:
fig, axes = plt.subplots(
    1,
    len(benchmark_scenarios),
    figsize=(6 * len(benchmark_scenarios), 4),
    constrained_layout=True,
)
axes = [axes] if len(benchmark_scenarios) == 1 else axes
for ax, scenario_name in zip(axes, benchmark_scenarios):
    group = benchmark_predictions.loc[
        (benchmark_predictions.scenario == scenario_name)
        & benchmark_predictions.split.eq("test")
    ]
    ax.plot(group.ds, group.y, "ko-", ms=3, label="Observed")
    ax.plot(group.ds, group.prediction, label="Prophet")
    ax.set(title=f"{scenario_name} final 28-day test", ylabel="Units")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
fig.savefig(benchmark_out / "final_test_predictions.png", dpi=150)
plt.show()
plt.close(fig)

required = {
    "metrics.csv",
    "predictions.csv",
    "tuning_results.csv",
    "selected_parameters.csv",
    "fit_details.csv",
    "explanations.csv",
    "manifest.json",
}
assert required.issubset({path.name for path in benchmark_out.iterdir()})
for name, digest in INPUT_HASHES.items():
    assert hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() == digest
print("Standard benchmark exports:", benchmark_out.resolve())